# All-variable control (`fixed_p = 0`)

The first experiment is the **control**: train on a constant C10 (one cyclic group of order 10,
held fixed every run) with `fixed_p = 0`, so *every* element-symbol is reshuffled each run.

With no stable symbol→element mapping, there is nothing to memorize — the model can only solve
**symbolically** (read this run's mapping from the in-context examples). So we expect it to
learn the group algorithm and generalize to the held-out fact, with **no grokking**: because
there is no memorization shortcut, there is no memorize-first / generalize-late gap. This is the
point of the control — it shows grokking is *absent* when the shortcut is removed.

Grokking proper needs a shortcut *and* something to generalize to, which only happens at
**intermediate `fixed_p`** (the follow-up: the `train_c10.sh` sweep). Weight decay is left on
here so that `fixed_p` is the only thing that differs between this control and those runs.

The readout is `symbolic_reliance` = how much held-out accuracy collapses when we relabel the
**context** symbols but leave the query intact. For an all-variable model it should rise *with*
`acc_full` and stay high — no late jump.

> Short proof-of-concept below (small model, 20k steps) to validate the pipeline and the early
> dynamics; run it on the Bau cluster (A100 / H100). The full-scale run is `experiments/train_c10.sh 0.0`.

In [ ]:
import os, subprocess
# Run from the repo root, whether launched inside the repo (cluster) or fresh (Colab).
if not os.path.exists('experiments/train_fixed_p.py'):
    if not os.path.isdir('algebra-grok'):
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/tohuya6/algebra-grok.git'], check=True)
    os.chdir('algebra-grok')
import torch
print('cwd:', os.getcwd())
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(cpu)')

## Train the control

`fixed_p=0.0` on the constant-C10 config (`--num_symbols 16 --max_order 10 --min_order 10
--mix 0`, already the script defaults) with the grokking hyperparameters
(`weight_decay=2.0`, `lr=1.5e-4`). Device defaults to `auto` (CUDA when present).

In [ ]:
!python experiments/train_fixed_p.py \
    --name control-p0 --fixed_p 0.0 \
    --task_name mixcyclic --weight_decay 2.0 --lr 1.5e-4 --bf16 \
    --d_model 128 --n_layers 2 --n_heads 4 --block_size 512 \
    --k_shots 100 --batch_size 128 --n_steps 20000 --evaluation_steps 1000

## Read the dynamics

The trainer writes `outputs/control-p0/metrics.json` every `evaluation_steps`. On a log-x axis:

- **acc_full** — held-out accuracy.
- **symbolic_reliance** — the generalization signal. For the all-variable control it should track
  `acc_full` upward and stay high, with **no late jump** — i.e. no grokking gap. (When `acc_full`
  is still ~0 early on, `symbolic_reliance` is undefined/noisy; that is expected.)

In [ ]:
import json
import matplotlib.pyplot as plt

h = [r for r in json.load(open('outputs/control-p0/metrics.json')) if r['step'] > 0]
step = [r['step'] for r in h]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(step, [r['acc_full'] for r in h])
ax[1].plot(step, [r['symbolic_reliance'] for r in h])
loss = [(r['step'], r['train_loss']) for r in h if r['train_loss'] is not None]
ax[2].plot([s for s, _ in loss], [l for _, l in loss])

for a, title, ylabel in zip(ax,
        ['acc_full (held-out accuracy)', 'symbolic_reliance', 'train_loss'],
        ['acc_full', 'reliance', 'loss']):
    a.set(title=title, xlabel='step', ylabel=ylabel)
    a.set_xscale('log'); a.grid(True, alpha=0.3)
plt.suptitle('fixed_p = 0 (all-variable control)')
plt.tight_layout(); plt.show()

## Next

This control establishes the no-grok baseline. To look for grokking, sweep into intermediate
`fixed_p` on a GPU/cluster, where a memorization shortcut and held-out generalization coexist:

```bash
for p in 0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9; do bash experiments/train_c10.sh $p; done
```

then compare the `symbolic_reliance` trajectories against this `fixed_p=0` control.
`verify_solution.ipynb` covers the static readout (validated on the released reference model).